# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Properties script
**Goal:** To train a surrogate model on a CSV dataset resulting from structural analyses done in grasshopper, this surrogate model should be able to accurately tell if a beam in the structure is structurally failing or not   
**Inputs:**
*   CSV with structural properties, from grasshopper

**Outputs:**
*   A surrogate model????

# PARAMETERS

In [1]:
import config

USE_DATA_4 = True  # Zet op True om data_4.csv te gebruiken
CSV_FILE = 'data_4.csv' if USE_DATA_4 else 'data_3.csv'
print(f"Actieve dataset: {CSV_FILE}")

# Training runtime opties
USE_TRAINING_TIME_LIMIT = False  # Zet op True voor korte test-runs
TRAINING_TIME_LIMIT_SECONDS = 60

# BENAMING VAN BESTAND EN NETWERK ARCHITECTUUR
schone_naam = CSV_FILE.replace('.csv', '')
delen = schone_naam.split('_')

OPTIMIZATION_TYPE = delen[0] if len(delen) > 0 else 'UNKNOWN'
DATE = delen[1] if len(delen) > 1 else '000000'
ITERATION = delen[2] if len(delen) > 2 else '0000'

prefix_sm = f"{OPTIMIZATION_TYPE}_{DATE}_{ITERATION}"
with open(config.SM_EXPORT_PATH / 'prefix_sm.txt', 'w') as f:
    f.write(prefix_sm)

✅ Systeem succesvol geladen.
📂 Code draait lokaal vanuit: thesis_generative_timber
☁️ Data gekoppeld aan OneDrive: 2.2 - 2.4
Actieve dataset: data_4.csv


# IMPORTING DATA

In [2]:
import config
import pandas as pd
import torch
import json
import re

print("1. Dataset inladen...")
df = pd.read_csv(config.GH_DATA_PATH / CSV_FILE)
print(f"✅ Dataset '{CSV_FILE}' ingeladen.")

with open(config.DATA_IO_PATH / 'edge_index.json', 'r') as f:
    edge_index = torch.tensor(json.load(f), dtype=torch.long)

# 2. Harde validatie van de dimensies
print("\n--- DATA VALIDATIE ---")
print(f"Aantal samples (rijen) in dataset: {len(df)}")
print(f"Totaal aantal kolommen in dataset: {df.shape[1]}")
print(f"Dimensie van de edge_index tensor: {edge_index.shape}")

# Dynamisch node- en edge-aantal afleiden uit de data
NODE_IDS = sorted({int(m.group(1)) for col in df.columns if (m := re.match(r'^v(\d+)_x$', col))})
assert NODE_IDS, "Fout: geen node-coordinaten gevonden in CSV (verwacht kolommen zoals v0_x)."
NODE_COUNT = len(NODE_IDS)

EDGE_COLS = sorted(
    [col for col in df.columns if re.match(r'^beam\d+_a_force$', col)],
    key=lambda c: int(re.match(r'^beam(\d+)_a_force$', c).group(1))
)
EDGE_COUNT = len(EDGE_COLS)

# Korte sanity check: detecteer automatisch of support/load features aanwezig zijn
for i in NODE_IDS:
    assert f"v{i}_y" in df.columns and f"v{i}_z" in df.columns, f"Fout: missende y/z kolommen voor vertex v{i}."

has_support_cols = all(f"v{i}_is_support" in df.columns for i in NODE_IDS)
has_load_cols = all(f"v{i}_load_z" in df.columns for i in NODE_IDS)
assert has_support_cols == has_load_cols, "Fout: onvolledige node-features. Verwacht per vertex zowel is_support als load_z."
NODE_FEATURE_DIM = 5 if (has_support_cols and has_load_cols) else 3
expected_columns = 1 + (NODE_COUNT * NODE_FEATURE_DIM) + EDGE_COUNT  # +1 voor sample_id
assert df.shape[1] == expected_columns, f"Fout: CSV heeft {df.shape[1]} kolommen, verwacht {expected_columns} op basis van schema."
assert edge_index.shape[0] == 2, "Fout: edge_index moet precies 2 rijen hebben (source en target nodes)."
assert edge_index.shape[1] == EDGE_COUNT, f"Fout: edge_index moet precies {EDGE_COUNT} kolommen (verbindingen) hebben."

print(f"Node count gedetecteerd: {NODE_COUNT}")
print(f"Edge count gedetecteerd: {EDGE_COUNT}")
print(f"Node feature dim gedetecteerd: {NODE_FEATURE_DIM}")
print("Validatie succesvol. Data is correct ingeladen.")

1. Dataset inladen...
✅ Dataset 'data_4.csv' ingeladen.

--- DATA VALIDATIE ---
Aantal samples (rijen) in dataset: 10000
Totaal aantal kolommen in dataset: 98
Dimensie van de edge_index tensor: torch.Size([2, 32])
Node count gedetecteerd: 13
Edge count gedetecteerd: 32
Node feature dim gedetecteerd: 5
Validatie succesvol. Data is correct ingeladen.


# PROCESSING DATA

In [3]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import os

# 1. Normalisatie (Scaling) instellen
# Neurale netwerken werken wiskundig het best met getallen rond de 0.
print("Start data normalisatie...")
node_scaler = StandardScaler()
edge_scaler = StandardScaler()

# Kolomnamen vooraf opbouwen
if NODE_FEATURE_DIM == 5:
    # Voor data_4: x, y, z, load_z en is_support per vertex
    node_cols = [f"v{i}_{feat}" for i in NODE_IDS for feat in ("x", "y", "z", "load_z", "is_support")]
else:
    node_cols = [f"v{i}_{axis}" for i in NODE_IDS for axis in ("x", "y", "z")]
edge_cols = EDGE_COLS

# 2. Ruwe data in 1x vectorized stap ophalen en herschalen
node_raw = df[node_cols].to_numpy(dtype=np.float32).reshape(-1, NODE_COUNT, NODE_FEATURE_DIM)
edge_raw = df[edge_cols].to_numpy(dtype=np.float32).reshape(-1, EDGE_COUNT, 1)

if NODE_FEATURE_DIM == 5:
    # Schaal alleen continue features; houd binaire support feature op 0/1
    node_continuous = node_raw[:, :, :4]
    node_support = node_raw[:, :, 4:5]
    node_scaler.fit(node_continuous.reshape(-1, 4))
    node_continuous_scaled = node_scaler.transform(node_continuous.reshape(-1, 4)).reshape(-1, NODE_COUNT, 4)
    node_scaled = np.concatenate([node_continuous_scaled, node_support], axis=2)
else:
    node_scaler.fit(node_raw.reshape(-1, 3))
    node_scaled = node_scaler.transform(node_raw.reshape(-1, 3)).reshape(-1, NODE_COUNT, 3)
edge_scaler.fit(edge_raw.reshape(-1, 1))

edge_scaled = edge_scaler.transform(edge_raw.reshape(-1, 1)).reshape(-1, EDGE_COUNT, 1)

# 3. Dataset opbouwen met de genormaliseerde data
print("Bouwen van Graph objecten...")
graph_dataset = [
    Data(
        x=torch.from_numpy(node_scaled[i]),
        edge_index=edge_index,
        y_edge=torch.from_numpy(edge_scaled[i])
    )
    for i in range(len(df))
]

# 4. Splitsen in Train (80%) en Test (20%) met vaste seed voor reproduceerbaarheid
rng = np.random.default_rng(42)
indices = rng.permutation(len(graph_dataset))
train_size = int(0.8 * len(graph_dataset))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

train_dataset = [graph_dataset[i] for i in train_idx]
test_dataset = [graph_dataset[i] for i in test_idx]

# 5. DataLoaders tunen voor efficiëntere input pipeline
loader_kwargs = {
    "batch_size": 32,
    "num_workers": min(4, os.cpu_count() or 1),
    "pin_memory": torch.cuda.is_available()
}
if loader_kwargs["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f"Dataset klaar! Train set: {len(train_dataset)} graphs. Test set: {len(test_dataset)} graphs.")

c:\Users\Jasper\Documents\PyEnvs\thesis_home_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Start data normalisatie...
Bouwen van Graph objecten...
Dataset klaar! Train set: 8000 graphs. Test set: 2000 graphs.


# MODEL SETUP

In [4]:
from c21_surrogate_model import TrussEdgeGNN
import torch

# Reset het model op je device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TrussEdgeGNN(node_in_dim=NODE_FEATURE_DIM, hidden_dim=128).to(device)

print(f"GNN model succesvol ingeladen op: {device}")
print("\n--- MODEL ARCHITECTUUR ---")
print(model)

GNN model succesvol ingeladen op: cpu

--- MODEL ARCHITECTUUR ---
TrussEdgeGNN(
  (conv1): SAGEConv(5, 128, aggr=mean)
  (conv2): SAGEConv(128, 128, aggr=mean)
  (conv3): SAGEConv(128, 128, aggr=mean)
  (conv4): SAGEConv(128, 128, aggr=mean)
  (edge_predictor): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


# MODEL TRAINING

In [5]:
from sklearn.metrics import r2_score
import time

# ==========================================
# 1. SETUP VAN DE OPTIMIZER EN LOSS FUNCTIE
# ==========================================
# lr=0.001 is je startwaarde. Pas deze aan als de Loss explodeert of stagneert.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Mean Squared Error (MSE) is de standaard voor regressie (het voorspellen van continue getallen)
criterion = torch.nn.MSELoss()

EPOCHS = 100
EVAL_EVERY = 10
train_start_time = time.time()

if USE_TRAINING_TIME_LIMIT:
    print(f"Start training voor {EPOCHS} epochs (max {TRAINING_TIME_LIMIT_SECONDS}s)...\n")
else:
    print(f"Start training voor {EPOCHS} epochs (zonder tijdslimiet)...\n")

# ==========================================
# 2. DE TRAINING LOOP
# ==========================================
for epoch in range(EPOCHS):
    if USE_TRAINING_TIME_LIMIT:
        elapsed = time.time() - train_start_time
        if elapsed >= TRAINING_TIME_LIMIT_SECONDS:
            print(f"Stop training na {elapsed:.1f}s (tijdslimiet bereikt).")
            break

    model.train() # Zet model in trainingsmodus
    total_loss = 0.0

    # Loop over alle batches in de trainingsdata
    for batch in train_loader:
        batch = batch.to(device, non_blocking=True)

        # Reset de gradiënten (wiskundige stappen) van de vorige iteratie
        optimizer.zero_grad(set_to_none=True)

        # Forward pass: Laat het model voorspellen
        out = model(batch.x, batch.edge_index)

        # Bereken de fout (Loss)
        loss = criterion(out, batch.y_edge)

        # Backward pass: Bereken hoe de gewichten moeten veranderen
        loss.backward()

        # Optimizer step: Pas de gewichten daadwerkelijk aan
        optimizer.step()

        # Tel de loss op voor de statistieken
        total_loss += loss.item() * batch.num_graphs

    avg_train_loss = total_loss / len(train_dataset)

    # ==========================================
    # 3. DE EVALUATIE LOOP (Elke EVAL_EVERY epochs)
    # ==========================================
    if (epoch + 1) % EVAL_EVERY == 0:
        model.eval() # Zet model in testmodus (geen gewichten aanpassen)
        pred_batches = []
        true_batches = []

        with torch.no_grad(): # Bespaar geheugen, we trainen hier niet
            for batch in test_loader:
                batch = batch.to(device, non_blocking=True)
                out = model(batch.x, batch.edge_index)

                pred_batches.append(out.detach().cpu())
                true_batches.append(batch.y_edge.detach().cpu())

        preds_scaled = torch.cat(pred_batches, dim=0).numpy()
        trues_scaled = torch.cat(true_batches, dim=0).numpy()

        # Transformeer de geschaalde getallen terug naar echte kiloNewtons (kN)
        preds_original = edge_scaler.inverse_transform(preds_scaled)
        trues_original = edge_scaler.inverse_transform(trues_scaled)

        # Bereken de R2 score op de echte (ongeziene) test data
        r2 = r2_score(trues_original, preds_original)
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss (Genormaliseerd): {avg_train_loss:.4f} | Test R2 Score: {r2:.4f}")

print("\nTraining afgerond! Je Graph Neural Network is klaar voor gebruik in je workflow.")

Start training voor 100 epochs (zonder tijdslimiet)...

Epoch 010/100 | Train Loss (Genormaliseerd): 0.1291 | Test R2 Score: 0.8635
Epoch 020/100 | Train Loss (Genormaliseerd): 0.1048 | Test R2 Score: 0.8831
Epoch 030/100 | Train Loss (Genormaliseerd): 0.0956 | Test R2 Score: 0.8908
Epoch 040/100 | Train Loss (Genormaliseerd): 0.0877 | Test R2 Score: 0.8927
Epoch 050/100 | Train Loss (Genormaliseerd): 0.0821 | Test R2 Score: 0.8968
Epoch 060/100 | Train Loss (Genormaliseerd): 0.0781 | Test R2 Score: 0.8987
Epoch 070/100 | Train Loss (Genormaliseerd): 0.0740 | Test R2 Score: 0.8986
Epoch 080/100 | Train Loss (Genormaliseerd): 0.0717 | Test R2 Score: 0.8983
Epoch 090/100 | Train Loss (Genormaliseerd): 0.0690 | Test R2 Score: 0.8958
Epoch 100/100 | Train Loss (Genormaliseerd): 0.0654 | Test R2 Score: 0.8942

Training afgerond! Je Graph Neural Network is klaar voor gebruik in je workflow.


# EXPORT

In [6]:
import joblib
import torch

# Exporteer de scalers die in deze notebook echt gebruikt zijn
node_scaler_path = config.SM_EXPORT_PATH / f"node_scaler_{prefix_sm}.pkl"
edge_scaler_path = config.SM_EXPORT_PATH / f"edge_scaler_{prefix_sm}.pkl"
joblib.dump(node_scaler, node_scaler_path)
joblib.dump(edge_scaler, edge_scaler_path)

print(f"\nScalers succesvol opgeslagen:")
print(f"- {node_scaler_path}")
print(f"- {edge_scaler_path}")

# Exporteer het getrainde GNN model als state_dict (PyTorch best practice)
model_path = config.SM_EXPORT_PATH / f"truss_edge_gnn_{prefix_sm}.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "node_in_dim": NODE_FEATURE_DIM,
        "hidden_dim": 128,
        "edge_count": EDGE_COUNT,
        "checkpoint_prefix": prefix_sm
    },
    model_path
)

print(f"\nModel checkpoint succesvol opgeslagen:")
print(f"- {model_path}")


Scalers succesvol opgeslagen:
- C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports\01_surrogate_models\node_scaler_data_4_0000.pkl
- C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports\01_surrogate_models\edge_scaler_data_4_0000.pkl

Model checkpoint succesvol opgeslagen:
- C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports\01_surrogate_models\truss_edge_gnn_data_4_0000.pt
